<a href="https://colab.research.google.com/github/mountaindew93/2026-1_CV/blob/main/HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# 실험 재현성을 위한 시드 고정
np.random.seed(42)
tf.random.set_seed(42)

In [2]:
# 1. Fashion-MNIST (실험 A, C)
(X_train_raw, y_train_raw), (X_val_raw, y_val_raw) = tf.keras.datasets.fashion_mnist.load_data()

# 정규화 및 평탄화
X_train_f = X_train_raw.reshape(-1, 784) / 255.0
X_val_f = X_val_raw.reshape(-1, 784) / 255.0
y_train_f = tf.one_hot(y_train_raw, 10)
y_val_f = tf.one_hot(y_val_raw, 10)

ds_train_f = tf.data.Dataset.from_tensor_slices((X_train_f, y_train_f)).shuffle(10000).batch(64)
ds_val_f = tf.data.Dataset.from_tensor_slices((X_val_f, y_val_f)).batch(64)

# 2. make_moons (실험 B - Dead ReLU 유도용)
X_m, y_m = make_moons(n_samples=1000, noise=0.15, random_state=42)
X_train_m, X_val_m, y_train_m, y_val_m = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

y_train_m = tf.one_hot(y_train_m, 2)
y_val_m = tf.one_hot(y_val_m, 2)

ds_train_m = tf.data.Dataset.from_tensor_slices((X_train_m, y_train_m)).shuffle(1000).batch(64)
ds_val_m = tf.data.Dataset.from_tensor_slices((X_val_m, y_val_m)).batch(64)

print("데이터셋 로드 완료!")

데이터셋 로드 완료!


In [3]:
# 실험 A, C에 사용할 기본 다층 퍼셉트론(MLP)
def get_mnist_model():
    inputs = tf.keras.Input(shape=(784,))
    x = tf.keras.layers.Dense(256, activation='relu')(inputs)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    outputs = tf.keras.layers.Dense(10)(x)
    return tf.keras.Model(inputs, outputs)

# 실험 B에 사용할 모델 (가중치를 매우 작게, 편향을 음수로 주어 고의로 ReLU를 죽임)
def get_activation_model(act_name):
    inputs = tf.keras.Input(shape=(2,))

    w_init = tf.keras.initializers.RandomNormal(stddev=0.01)
    b_init = tf.keras.initializers.Constant(-0.5)

    x1 = tf.keras.layers.Dense(64, kernel_initializer=w_init, bias_initializer=b_init)(inputs)
    act1 = tf.keras.layers.LeakyReLU(0.01)(x1) if act_name == 'leaky_relu' else tf.keras.layers.Activation(act_name)(x1)

    x2 = tf.keras.layers.Dense(64, kernel_initializer=w_init, bias_initializer=b_init)(act1)
    act2 = tf.keras.layers.LeakyReLU(0.01)(x2) if act_name == 'leaky_relu' else tf.keras.layers.Activation(act_name)(x2)

    outputs = tf.keras.layers.Dense(2)(act2)
    # 중간 활성화 분포를 보기 위해 act1, act2도 같이 리턴
    return tf.keras.Model(inputs, [act1, act2, outputs])

In [4]:
def run_exp_a(loss_type, epochs=30):
    print(f"\n--- 실험 A: {loss_type} 학습 시작 ---")
    tf.random.set_seed(42)
    model = get_mnist_model()
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True) if loss_type == 'Cross Entropy' else tf.keras.losses.MeanSquaredError()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'grad_mean': []}

    for epoch in range(epochs):
        loss_metric = tf.keras.metrics.Mean()
        acc_metric = tf.keras.metrics.CategoricalAccuracy()
        grads_epoch = []

        # Train
        for x, y in ds_train_f:
            with tf.GradientTape() as tape:
                logits = model(x, training=True)
                preds = tf.nn.softmax(logits)
                loss = loss_fn(y, logits) if loss_type == 'Cross Entropy' else loss_fn(y, preds)

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

            loss_metric.update_state(loss)
            acc_metric.update_state(y, preds)
            # 마지막 출력층의 기울기 크기 추적 (Vanishing 증명용)
            grads_epoch.append(tf.reduce_mean(tf.abs(grads[-2])).numpy())

        # Validation
        val_acc_metric = tf.keras.metrics.CategoricalAccuracy()
        val_loss_metric = tf.keras.metrics.Mean()
        for x_val, y_val in ds_val_f:
            val_logits = model(x_val, training=False)
            val_preds = tf.nn.softmax(val_logits)
            val_loss = loss_fn(y_val, val_logits) if loss_type == 'Cross Entropy' else loss_fn(y_val, val_preds)

            val_loss_metric.update_state(val_loss)
            val_acc_metric.update_state(y_val, val_preds)

        # 기록 저장
        history['train_loss'].append(loss_metric.result().numpy())
        history['train_acc'].append(acc_metric.result().numpy())
        history['val_loss'].append(val_loss_metric.result().numpy())
        history['val_acc'].append(val_acc_metric.result().numpy())
        history['grad_mean'].append(np.mean(grads_epoch))

        # 진행상황 출력
        print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {history['train_loss'][-1]:.4f} | Train Acc: {history['train_acc'][-1]*100:.2f}% | Val Acc: {history['val_acc'][-1]*100:.2f}%")

    return history

res_ce = run_exp_a('Cross Entropy', epochs=30)
res_mse = run_exp_a('MSE', epochs=30)


--- 실험 A: Cross Entropy 학습 시작 ---
Epoch 01/30 | Train Loss: 0.0258 | Train Acc: 82.03% | Val Acc: 85.08%
Epoch 02/30 | Train Loss: 0.0196 | Train Acc: 86.46% | Val Acc: 85.41%
Epoch 03/30 | Train Loss: 0.0178 | Train Acc: 87.68% | Val Acc: 86.70%
Epoch 04/30 | Train Loss: 0.0167 | Train Acc: 88.58% | Val Acc: 87.28%
Epoch 05/30 | Train Loss: 0.0158 | Train Acc: 89.21% | Val Acc: 87.24%
Epoch 06/30 | Train Loss: 0.0153 | Train Acc: 89.49% | Val Acc: 88.11%
Epoch 07/30 | Train Loss: 0.0146 | Train Acc: 90.02% | Val Acc: 88.15%
Epoch 08/30 | Train Loss: 0.0142 | Train Acc: 90.36% | Val Acc: 88.27%
Epoch 09/30 | Train Loss: 0.0137 | Train Acc: 90.72% | Val Acc: 88.32%
Epoch 10/30 | Train Loss: 0.0132 | Train Acc: 91.06% | Val Acc: 88.23%
Epoch 11/30 | Train Loss: 0.0128 | Train Acc: 91.45% | Val Acc: 88.68%
Epoch 12/30 | Train Loss: 0.0124 | Train Acc: 91.66% | Val Acc: 88.70%
Epoch 13/30 | Train Loss: 0.0121 | Train Acc: 91.99% | Val Acc: 87.76%
Epoch 14/30 | Train Loss: 0.0118 | Train A

In [1]:
epochs_range = range(1, EPOCHS_A + 1)

# --- 1. 메인 2x2 그리드 그래프 (Loss & Accuracy 비교) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# [1행] CrossEntropy 영역
axes[0, 0].plot(epochs_range, ce_history['train_loss'], label='Train Loss', color='#1f77b4')
axes[0, 0].plot(epochs_range, ce_history['val_loss'], label='Validation Loss', color='#ff7f0e')
axes[0, 0].set_title('Fashion-MNIST - CrossEntropy: Loss Curve')
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(epochs_range, ce_history['train_acc'], label='Train Accuracy', color='#1f77b4')
axes[0, 1].plot(epochs_range, ce_history['val_acc'], label='Validation Accuracy', color='#ff7f0e')
axes[0, 1].set_title('Fashion-MNIST - CrossEntropy: Accuracy Curve')
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True)

# [2행] MSE 영역
axes[1, 0].plot(epochs_range, mse_history['train_loss'], label='Train Loss', color='#1f77b4')
axes[1, 0].plot(epochs_range, mse_history['val_loss'], label='Validation Loss', color='#ff7f0e')
axes[1, 0].set_title('Fashion-MNIST - MSE: Loss Curve')
axes[1, 0].set_xlabel('Epochs')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(epochs_range, mse_history['train_acc'], label='Train Accuracy', color='#1f77b4')
axes[1, 1].plot(epochs_range, mse_history['val_acc'], label='Validation Accuracy', color='#ff7f0e')
axes[1, 1].set_title('Fashion-MNIST - MSE: Accuracy Curve')
axes[1, 1].set_xlabel('Epochs')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True)

# 레이아웃 자동 조정 후 여백 수동 확보 (텍스트가 들어갈 자리)
plt.tight_layout()
plt.subplots_adjust(top=0.88, hspace=0.6)

# 최상단 및 행별 텍스트 타이틀 삽입 (요청하신 사진과 동일한 포맷)
fig.text(0.5, 0.96, '<Loss vs Epochs>', ha='center', fontsize=20, fontweight='bold')
fig.text(0.5, 0.92, 'CrossEntropy', ha='center', fontsize=18, fontweight='bold')
fig.text(0.5, 0.89, 'Performance Metrics for CrossEntropy on Fashion-MNIST', ha='center', fontsize=14)

fig.text(0.5, 0.49, 'MSE', ha='center', fontsize=18, fontweight='bold')
fig.text(0.5, 0.46, 'Performance Metrics for MSE on Fashion-MNIST', ha='center', fontsize=14)

plt.show()

# --- 2. Gradient Flow 시각화 (과제 평가 기준 충족을 위한 별도 출력) ---
plt.figure(figsize=(8, 4))
plt.plot(epochs_range, ce_history['output_grad_mean'], 'b-o', label='CrossEntropy Grad')
plt.plot(epochs_range, mse_history['output_grad_mean'], 'r-o', label='MSE Grad')
plt.title('Output Layer Gradient Flow (Vanishing Gradient Check)')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Gradient')
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'EPOCHS_A' is not defined

In [ ]:
def run_exp_b(act_name, epochs=150):
    print(f"\n--- 실험 B: {act_name} 학습 시작 ---")
    tf.random.set_seed(42)
    model = get_activation_model(act_name)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
    history = {'val_acc': []}

    for epoch in range(epochs):
        val_acc = tf.keras.metrics.CategoricalAccuracy()

        for x, y in ds_train_m:
            with tf.GradientTape() as tape:
                _, _, logits = model(x, training=True)
                loss = loss_fn(y, logits)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

        for x_val, y_val in ds_val_m:
            _, _, val_logits = model(x_val, training=False)
            val_acc.update_state(y_val, tf.nn.softmax(val_logits))

        history['val_acc'].append(val_acc.result().numpy())

        # 150 에폭이므로 10단위로 축약해서 출력
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d}/{epochs} | Val Acc: {history['val_acc'][-1]*100:.2f}%")

    # 마지막 에폭에서 Layer 1의 활성화 값 전체 수집 (히트맵용)
    act1_out, _, _ = model(X_val_m, training=False)
    history['final_act'] = act1_out.numpy()

    return history

res_relu = run_exp_b('relu', epochs=150)
res_leaky = run_exp_b('leaky_relu', epochs=150)
res_sig = run_exp_b('sigmoid', epochs=150)

In [ ]:
# =====================================================================
# [실험 B]
# =====================================================================
plt.figure(figsize=(20, 12))
e_range = range(1, EPOCHS_B + 1)

plt.subplot(2, 3, 1)
plt.plot(e_range, res_relu['val_acc'], label='ReLU')
plt.plot(e_range, res_leaky['val_acc'], label='LeakyReLU')
plt.plot(e_range, res_sig['val_acc'], label='Sigmoid')
plt.title('Validation Accuracy over Epochs')
plt.legend(); plt.grid(True)

plt.subplot(2, 3, 2)
sns.kdeplot(res_relu['final_act1'].flatten(), fill=True, label='ReLU', color='blue', alpha=0.5)
sns.kdeplot(res_leaky['final_act1'].flatten(), fill=True, label='LeakyReLU', color='orange', alpha=0.5)
sns.kdeplot(res_sig['final_act1'].flatten(), fill=True, label='Sigmoid', color='green', alpha=0.5)
plt.title('Layer 1 Activation Distribution'); plt.legend()

def plot_dead_heatmap(act_data, ax, title, is_sigmoid=False):
    ratio = np.mean((act_data <= 0.1) | (act_data >= 0.9), axis=0) if is_sigmoid else np.mean(act_data <= 0.0, axis=0)
    cmap = "Purples" if is_sigmoid else "Reds"
    sns.heatmap(ratio.reshape(8, 8), annot=True, fmt=".1f", cmap=cmap, ax=ax, vmin=0, vmax=1)
    ax.set_title(title); ax.axis('off')

plot_dead_heatmap(res_relu['final_act1'], plt.subplot(2, 3, 4), 'ReLU: Dead Neuron Heatmap')
plot_dead_heatmap(res_leaky['final_act1'], plt.subplot(2, 3, 5), 'LeakyReLU: Negative Region Ratio')
plot_dead_heatmap(res_sig['final_act1'], plt.subplot(2, 3, 6), 'Sigmoid: Saturation Heatmap', is_sigmoid=True)

plt.tight_layout(); plt.show()

In [ ]:
def run_exp_c(opt_name, lr, epochs=20):
    print(f"\n--- 실험 C: {opt_name} (LR: {lr}) 학습 시작 ---")
    tf.random.set_seed(42)
    model = get_mnist_model()

    # 지수 감소 학습률 스케줄러 (매 에폭마다 0.9배씩 감소)
    steps_per_epoch = len(X_train_f) // 64
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr, decay_steps=steps_per_epoch, decay_rate=0.9, staircase=True
    )

    if opt_name == 'SGD':
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)
    elif opt_name == 'Momentum':
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
    history = {'loss': [], 'val_acc': [], 'grad_mean': []}

    for epoch in range(epochs):
        loss_metric = tf.keras.metrics.Mean()
        val_acc = tf.keras.metrics.CategoricalAccuracy()
        grads_epoch = []

        for x, y in ds_train_f:
            with tf.GradientTape() as tape:
                logits = model(x, training=True)
                loss = loss_fn(y, logits)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

            loss_metric.update_state(loss)
            grads_epoch.append(tf.reduce_mean(tf.abs(grads[-2])).numpy())

        for x_val, y_val in ds_val_f:
            val_logits = model(x_val, training=False)
            val_acc.update_state(y_val, tf.nn.softmax(val_logits))

        history['loss'].append(loss_metric.result().numpy())
        history['val_acc'].append(val_acc.result().numpy())
        history['grad_mean'].append(np.mean(grads_epoch))

        print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {history['loss'][-1]:.4f} | Val Acc: {history['val_acc'][-1]*100:.2f}%")

    return history

opts = ['SGD', 'Momentum', 'Adam']
lrs = [0.1, 0.01, 0.001]
res_opt = {opt: {} for opt in opts}

for opt in opts:
    for lr in lrs:
        res_opt[opt][lr] = run_exp_c(opt, lr, epochs=20)

In [ ]:
plt.figure(figsize=(18, 12))
e_range = range(1, EPOCHS_C + 1)

plt.subplot(2, 2, 1)
plt.plot(e_range, results['SGD'][0.01]['train_loss'], label='SGD')
plt.plot(e_range, results['Momentum'][0.01]['train_loss'], label='Momentum')
plt.plot(e_range, results['Adam'][0.01]['train_loss'], label='Adam')
plt.title('Loss Comparison (LR=0.01)'); plt.legend(); plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(e_range, results['Adam'][0.1]['train_loss'], label='Adam LR=0.1')
plt.plot(e_range, results['Adam'][0.01]['train_loss'], label='Adam LR=0.01')
plt.plot(e_range, results['Adam'][0.001]['train_loss'], label='Adam LR=0.001')
plt.title('Adam Learning Rate Impact (Loss)'); plt.legend(); plt.grid(True)

plt.subplot(2, 2, 3)
plt.plot(e_range, results['SGD'][0.001]['val_acc'], label='SGD')
plt.plot(e_range, results['Momentum'][0.001]['val_acc'], label='Momentum')
plt.plot(e_range, results['Adam'][0.001]['val_acc'], label='Adam')
plt.title('Accuracy Comparison (LR=0.001)'); plt.legend(); plt.grid(True)

plt.subplot(2, 2, 4)
plt.plot(e_range, results['Adam'][0.1]['grad_mean'], label='LR=0.1 (Unstable)', linestyle='--')
plt.plot(e_range, results['Adam'][0.001]['grad_mean'], label='LR=0.001 (Stable)')
plt.title('Adam Gradient Flow Stability'); plt.legend(); plt.grid(True)

plt.tight_layout(); plt.show()